<a href="https://colab.research.google.com/github/cameronliddle/ThesisAIDetection/blob/main/EfficientNet_B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!cp -r /content/drive/MyDrive/Thesis/Processed_Dataset/resplit_dataset /content/


In [3]:
#imports
import timm
import torch
import torch.nn as nn
import numpy as np
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve, auc
import matplotlib.pyplot as plt
import json
from tqdm import tqdm


In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [5]:
base_path = '/content/resplit_dataset'
train_dir = f"{base_path}/train"
val_dir = f"{base_path}/val"
test_dir = f"{base_path}/test"

save_dir = '/content/drive/MyDrive/Thesis/ModelsColab/CNNS/EfficientNetB0'


In [6]:
effnet_b0 = timm.create_model('efficientnet_b0', pretrained=True, num_classes=1)
effnet_b0 = effnet_b0.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(effnet_b0.parameters(), lr=1e-4)


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [7]:
#transformers and dataloaders
img_size = (224, 224)
batch_size = 8

train_transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_test_transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])


In [8]:
train_dataset = datasets.ImageFolder(train_dir, transform=train_transform)
val_dataset = datasets.ImageFolder(val_dir, transform=val_test_transform)
test_dataset = datasets.ImageFolder(test_dir, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)


In [9]:
#train function
def train(model, loader, optimizer, criterion):
    model.train()
    total_loss = 0
    correct = 0
    total = 0
    for inputs, labels in tqdm(loader, desc="Training"):
        inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        preds = (torch.sigmoid(outputs) > 0.5).int()
        correct += (preds == labels.int()).sum().item()
        total += labels.size(0)

    accuracy = correct / total
    return total_loss / len(loader), accuracy


In [10]:
#evaluation function
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, total = 0, 0, 0
    all_labels, all_outputs = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device).float().unsqueeze(1)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            all_labels.extend(labels.cpu().numpy())
            all_outputs.extend(outputs.cpu().numpy())
            preds = (torch.sigmoid(outputs) > 0.5).int()
            correct += (preds == labels.int()).sum().item()
            total += labels.size(0)
            total_loss += loss.item()

    accuracy = correct / total
    f1 = f1_score(all_labels, (np.array(all_outputs) > 0.0).astype(int))
    precision = precision_score(all_labels, (np.array(all_outputs) > 0.0).astype(int))
    recall = recall_score(all_labels, (np.array(all_outputs) > 0.0).astype(int))
    return total_loss / len(loader), accuracy, f1, precision, recall, all_labels, all_outputs


In [11]:
train_losses, val_losses = [], []
train_accuracies, val_accuracies = [], []

epochs = 25
for epoch in range(epochs):
    train_loss, train_accuracy = train(effnet_b0, train_loader, optimizer, criterion)
    val_loss, val_accuracy, val_f1, val_precision, val_recall, _, _ = evaluate(effnet_b0, val_loader, criterion)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_accuracy)
    val_accuracies.append(val_accuracy)

    print(f"Epoch {epoch+1}/{epochs} => "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_accuracy*100:.2f}% | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_accuracy*100:.2f}% | "
          f"F1: {val_f1:.4f}")


Training: 100%|██████████| 1575/1575 [01:16<00:00, 20.71it/s]


Epoch 1/25 => Train Loss: 0.6481 | Train Acc: 83.03% | Val Loss: 0.3675 | Val Acc: 89.07% | F1: 0.8961


Training: 100%|██████████| 1575/1575 [01:14<00:00, 21.25it/s]


Epoch 2/25 => Train Loss: 0.3409 | Train Acc: 88.97% | Val Loss: 0.2202 | Val Acc: 92.07% | F1: 0.9261


Training: 100%|██████████| 1575/1575 [01:14<00:00, 21.16it/s]


...

Epoch 23/25 => Train Loss: 0.0303 | Train Acc: 99.02% | Val Loss: 0.1313 | Val Acc: 96.67% | F1: 0.9670


Training: 100%|██████████| 1575/1575 [01:14<00:00, 21.19it/s]


Epoch 24/25 => Train Loss: 0.0230 | Train Acc: 99.34% | Val Loss: 0.1156 | Val Acc: 97.03% | F1: 0.9708


Training: 100%|██████████| 1575/1575 [01:14<00:00, 21.18it/s]


Epoch 25/25 => Train Loss: 0.0261 | Train Acc: 99.09% | Val Loss: 0.1086 | Val Acc: 96.97% | F1: 0.9705


In [12]:
test_loss, test_accuracy, test_f1, test_precision, test_recall, test_labels, test_outputs = evaluate(effnet_b0, test_loader, criterion)


In [13]:
torch.save(effnet_b0.state_dict(), f"{save_dir}/efficientnet_b0.pth")


In [14]:
training_history = {
    'train_loss': train_losses,
    'val_loss': val_losses,
    'train_accuracy': [acc * 100 for acc in train_accuracies],
    'val_accuracy': [acc * 100 for acc in val_accuracies]
}
with open(f"{save_dir}/efficientnet_b0_training_history.json", 'w') as f:
    json.dump(training_history, f, indent=4)


In [15]:
final_results = {
    'Validation Accuracy (%)': round(val_accuracies[-1] * 100, 2),
    'Validation Loss': round(val_losses[-1], 4),
    'Test Accuracy (%)': round(test_accuracy * 100, 2),
    'Test Loss': round(test_loss, 4),
    'Test F1-Score': round(test_f1, 4),
    'Test Precision': round(test_precision, 4),
    'Test Recall': round(test_recall, 4)
}
with open(f"{save_dir}/efficientnet_b0_results.json", 'w') as f:
    json.dump(final_results, f, indent=4)


In [16]:
#Loss curve
plt.figure()
plt.plot(range(1, epochs+1), train_losses, label='Train Loss', marker='o')
plt.plot(range(1, epochs+1), val_losses, label='Validation Loss', marker='o')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('EfficientNet-B0 Loss Curve')
plt.legend()
plt.grid(True)
plt.savefig(f"{save_dir}/efficientnet_b0_loss_curve.png")
plt.close()


In [17]:
# accuracy curve
plt.figure()
plt.plot(range(1, epochs+1), [acc * 100 for acc in train_accuracies], label='Train Accuracy', marker='o', color='blue')
plt.plot(range(1, epochs+1), [acc * 100 for acc in val_accuracies], label='Validation Accuracy', marker='o', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('EfficientNet-B0 Accuracy Curve')
plt.legend()
plt.grid(True)
plt.savefig(f"{save_dir}/efficientnet_b0_accuracy_curve.png")
plt.close()


In [18]:
# confusion matrix
test_preds = (np.array(test_outputs) > 0.0).astype(int)
conf_matrix = confusion_matrix(test_labels, test_preds)
cmd = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=["Fake", "Real"])
cmd.plot(cmap='Blues')
plt.title('EfficientNet-B0 Confusion Matrix')
plt.savefig(f"{save_dir}/efficientnet_b0_confusion_matrix.png")
plt.close()


In [19]:
# roc curve
fpr, tpr, _ = roc_curve(test_labels, np.array(test_outputs))
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f"ROC Curve (AUC = {roc_auc:.2f})")
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('EfficientNet-B0 ROC Curve')
plt.legend(loc="lower right")
plt.grid(True)
plt.savefig(f"{save_dir}/efficientnet_b0_roc_curve.png")
plt.close()


In [20]:
print(" All efficientnet_b0 files saved successfully!")


 All efficientnet_b0 files saved successfully!
